# JASPAR — Transcription Factor Binding Profiles — Data Ingestion

**JASPAR** is the largest open-access database of curated, non-redundant transcription factor (TF) binding profiles, stored as **position frequency matrices (PFMs)** and derived **position weight matrices (PWMs)**. Maintained at the Centre for Molecular Medicine Norway (NCMM) and hosted via mirrors at EBI and elsewhere, it is one of the ELIXIR Core Data Resources and the canonical reference used by virtually every tool that scores DNA sequences for TF binding.

Each profile is produced from experimentally-determined binding sites (ChIP-seq, SELEX, HT-SELEX, PBM, etc.) and records — for every base pair of the binding motif — how many times each of the four nucleotides (A, C, G, T) was observed. From this matrix one can derive an information-content logo, a log-odds PWM for scoring new sequences, and a background-adjusted probability model.

Key data types provided by JASPAR:

| Field | Description |
|---|---|
| `matrix_id` | Stable profile accession with version, e.g. `MA0106.3` |
| `name` | Transcription factor gene symbol, e.g. `TP53` |
| `collection` | One of `CORE`, `CNE`, `PHYLOFACTS`, `SPLICE`, `POLII`, `FAM`, `PBM`, `PBM_HOMEO`, `PBM_HLH`, `UNVALIDATED` |
| `tax_group` | Broad taxonomic group: `vertebrates`, `plants`, `insects`, `nematodes`, `fungi`, `urochordates` |
| `species` | NCBI taxon ID(s) for the source organism(s) |
| `tf_family` | DNA-binding domain family (e.g. `p53-like`) |
| `tf_class` | Higher-level structural class (e.g. `Loop-sheet-helix factors`) |
| `data_type` | Experimental assay: `ChIP-seq`, `SELEX`, `HT-SELEX`, `PBM`, etc. |
| `pfm` | Position frequency matrix — dict of four lists (A/C/G/T) of length `L` |
| `version` | Profile version (matrix IDs are re-versioned when the model is updated) |

**REST API base:** `https://jaspar.genereg.net/api/v1/`

**Reference:** Rauluseviciute et al. (2024), *Nucleic Acids Research*, JASPAR 2024: 20th anniversary of the open-access database of transcription factor binding profiles. https://doi.org/10.1093/nar/gkad1059

In [ ]:
import json
import time
from pathlib import Path

import requests
import numpy as np
import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to the JASPAR REST API and confirm access via `/` root
    * [x] Download matrix metadata with collection / tax-group filters
    * [x] Fetch the PFM for a specific matrix (e.g. `MA0106.3` — TP53) and parse into a numpy array
    * [x] Parse matrix metadata into a Polars DataFrame
    * [x] Save data to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise matrix counts by collection, tax_group, tf_family, and tf_class
    * [ ] Inspect distribution of motif lengths and data-generating assays (`data_type`)
    * [ ] Identify redundant / deprecated profile versions (same base `matrix_id`, different version)
* [ ] **Analysis**
    * [ ] Convert PFMs to PWMs (log-odds) using a uniform and GC-adjusted background
    * [ ] Compute per-position information content I(i) = 2 - H(i) in bits
    * [ ] Compare motif similarity across TF families via Pearson correlation / Tomtom-style scoring
* [ ] **Visualization**
    * [ ] Sequence logos coloured by nucleotide
    * [ ] Heatmap of pairwise motif similarity within a TF family
    * [ ] Stacked bar of matrix counts by tax_group × collection
* [ ] **Statistical analysis**
    * [ ] Discuss pseudocount choice when converting PFM → PWM and its effect on log-odds tails
    * [ ] Error propagation from finite PFM sample sizes to information-content estimates
    * [ ] Multiple-testing correction for genome-wide PWM scanning (per-sequence FDR vs family-wise error)

## 1. Ingest Data

### 1.1 Connect to the JASPAR REST API

In [ ]:
JASPAR_API_BASE = "https://jaspar.genereg.net/api/v1"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)  # ensure the local cache directory exists

# JASPAR is a Django REST Framework service — by default it returns HTML for
# browser clients. An explicit Accept header forces JSON responses.
HEADERS = {"Accept": "application/json"}


def jaspar_get(endpoint: str, params: dict | None = None) -> dict:
    """
    Send a GET request to the JASPAR REST API and return parsed JSON.

    Parameters
    ----------
    endpoint : str
        Path relative to ``JASPAR_API_BASE``, e.g. ``"matrix/"`` or
        ``"matrix/MA0106.3/"``. A leading slash is optional.
    params : dict, optional
        Query-string parameters such as ``{"collection": "CORE",
        "tax_group": "vertebrates", "page_size": 100}``.

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    url = f"{JASPAR_API_BASE}/{endpoint.lstrip('/')}"
    resp = requests.get(url, headers=HEADERS, params=params or {}, timeout=30)
    resp.raise_for_status()
    time.sleep(0.2)  # polite delay — JASPAR is community-hosted
    return resp.json()


# ── Connectivity check: the API root lists the available resources ──────────
root = jaspar_get("")

print("JASPAR REST API — root endpoint")
print("=" * 50)
# The root returns a dict mapping resource names -> absolute URLs
for name, link in root.items():
    print(f"  {name:<18} {link}")

### 1.2 Download Matrix Metadata (CORE Vertebrate Collection)

The `/matrix/` endpoint is paginated (DRF cursor/page pagination). We collect all pages by walking the `next` link until it is `null`. Filtering by `collection=CORE` and `tax_group=vertebrates` restricts the download to the most widely-used human/mouse/zebrafish/etc. profiles.

In [ ]:
MATRIX_LIST_CACHE = DATA_DIR / "jaspar_core_vertebrates.json"


def fetch_all_matrices(
    collection: str = "CORE",
    tax_group: str = "vertebrates",
    page_size: int = 100,
    cache_path: Path | None = None,
) -> list[dict]:
    """
    Download every matrix record matching the given collection / tax_group.

    The JASPAR matrix-list endpoint is paginated; each page JSON contains
    ``count``, ``next``, ``previous``, and ``results``. We iterate pages by
    following the ``next`` URL until it becomes ``None`` and concatenate the
    ``results`` arrays.

    Parameters
    ----------
    collection : str, default "CORE"
        JASPAR collection. CORE is the curated, non-redundant collection.
    tax_group : str, default "vertebrates"
        Broad taxonomic group filter. One of ``vertebrates``, ``plants``,
        ``insects``, ``nematodes``, ``fungi``, ``urochordates``.
    page_size : int, default 100
        Results per page. JASPAR caps this server-side (typically at 100).
    cache_path : Path, optional
        If given and the file exists, return the cached JSON instead of
        hitting the API. Fresh downloads are written to this path.

    Returns
    -------
    list of dict
        Flat list of matrix metadata records. Each record has keys such as
        ``matrix_id``, ``name``, ``collection``, ``url`` (to fetch the PFM).
    """
    # ── Cache short-circuit ──────────────────────────────────────────────────
    if cache_path is not None and cache_path.exists():
        print(f"Cache hit: {cache_path}  ({cache_path.stat().st_size / 1024:.1f} KB)")
        return json.loads(cache_path.read_text())

    # ── Walk paginated results ───────────────────────────────────────────────
    params = {
        "collection": collection,
        "tax_group": tax_group,
        "page_size": page_size,
    }
    page = jaspar_get("matrix/", params=params)
    all_records: list[dict] = list(page["results"])
    total = page.get("count", len(all_records))
    print(f"Total matching matrices: {total}")

    # Follow the "next" link — it is an absolute URL so we call requests.get
    # directly rather than routing it through jaspar_get()
    next_url = page.get("next")
    while next_url:
        resp = requests.get(next_url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        time.sleep(0.2)
        page = resp.json()
        all_records.extend(page["results"])
        next_url = page.get("next")
        print(f"  fetched {len(all_records):>5} / {total}")

    # ── Write to cache ───────────────────────────────────────────────────────
    if cache_path is not None:
        cache_path.write_text(json.dumps(all_records, indent=2))
        print(f"Saved to: {cache_path}")

    return all_records


matrix_records = fetch_all_matrices(
    collection="CORE",
    tax_group="vertebrates",
    cache_path=MATRIX_LIST_CACHE,
)

# Show the keys available on a single metadata record
print(f"\nFields on a matrix-list record: {sorted(matrix_records[0].keys())}")
print(f"First record: {matrix_records[0]}")

### 1.3 Fetch a Single Matrix and Parse its PFM into a NumPy Array

The detailed matrix endpoint `/matrix/{matrix_id}/` returns the position frequency matrix under a `pfm` key, shaped as a dict:

```json
{"A": [n_A(1), n_A(2), ..., n_A(L)],
 "C": [...], "G": [...], "T": [...]}
```

where `n_X(i)` is the number of times base `X` was observed at position `i` across all training sites. We use `MA0106.3` — the TP53 tumour-suppressor profile — as a recognisable example.

In [ ]:
# Canonical alphabet ordering for PFMs throughout this notebook.
# Using a fixed order lets downstream code index rows as ACGT = 0,1,2,3.
PFM_ALPHABET = ("A", "C", "G", "T")


def fetch_matrix_detail(matrix_id: str, cache_dir: Path = DATA_DIR / "matrices") -> dict:
    """
    Fetch the full record for a single JASPAR matrix, with on-disk caching.

    Parameters
    ----------
    matrix_id : str
        Versioned JASPAR accession, e.g. ``"MA0106.3"``.
    cache_dir : Path
        Directory in which to cache individual matrix JSON responses. One
        file is written per matrix (``{matrix_id}.json``).

    Returns
    -------
    dict
        Full matrix record including ``pfm``, ``name``, ``tf_family``,
        ``tf_class``, ``data_type``, ``species``, etc.
    """
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{matrix_id}.json"

    if cache_path.exists():
        return json.loads(cache_path.read_text())

    record = jaspar_get(f"matrix/{matrix_id}/")
    cache_path.write_text(json.dumps(record, indent=2))
    return record


def pfm_to_numpy(pfm: dict, alphabet: tuple = PFM_ALPHABET) -> np.ndarray:
    """
    Convert a JASPAR PFM dict to a (4, L) NumPy array.

    Parameters
    ----------
    pfm : dict
        JASPAR ``pfm`` field: a mapping with keys ``"A"``, ``"C"``, ``"G"``,
        ``"T"`` whose values are equal-length lists of non-negative counts.
    alphabet : tuple of str, default ``("A", "C", "G", "T")``
        Row ordering for the output array.

    Returns
    -------
    np.ndarray
        Array of shape ``(4, L)`` and dtype ``float64``. Row ``i`` holds the
        counts for ``alphabet[i]``. Float dtype is chosen so that downstream
        normalisation and log-odds calculations do not need to re-cast.

    Raises
    ------
    ValueError
        If the alphabet rows have inconsistent lengths or a base is missing.
    """
    # Validate that every requested base is present in the payload
    missing = [b for b in alphabet if b not in pfm]
    if missing:
        raise ValueError(f"PFM is missing bases: {missing}")

    # Stack rows in canonical order. np.asarray preserves the row lengths.
    rows = [np.asarray(pfm[b], dtype=np.float64) for b in alphabet]

    # All rows must have identical length L (columns = motif positions)
    lengths = {len(r) for r in rows}
    if len(lengths) != 1:
        raise ValueError(f"PFM rows have inconsistent lengths: {lengths}")

    return np.vstack(rows)


# ── Example: fetch TP53 (MA0106.3) and parse its PFM ────────────────────────
tp53 = fetch_matrix_detail("MA0106.3")

print(f"Matrix ID   : {tp53.get('matrix_id')}")
print(f"Name        : {tp53.get('name')}")
print(f"Collection  : {tp53.get('collection')}")
print(f"TF family   : {tp53.get('tf_family')}")
print(f"TF class    : {tp53.get('tf_class')}")
print(f"Data type   : {tp53.get('data_type')}")
print(f"Species     : {tp53.get('species')}")

pfm_array = pfm_to_numpy(tp53["pfm"])
print(f"\nPFM shape   : {pfm_array.shape}  (4 bases × {pfm_array.shape[1]} positions)")
print(f"PFM dtype   : {pfm_array.dtype}")
print(f"Column sum  : {pfm_array.sum(axis=0)[:5]} ...  (should be ~constant = # training sites)")
print(f"\nFirst 5 columns (A/C/G/T):\n{pfm_array[:, :5]}")

### 1.4 Parse Matrix Metadata into a Polars DataFrame

The list endpoint returns a compact per-matrix summary. We convert it into a Polars DataFrame, parse the `matrix_id` into its base accession and integer version (same base ID, higher version = newer profile), and cast low-cardinality columns to `Categorical`.

In [ ]:
METADATA_PARQUET = DATA_DIR / "jaspar_matrix_metadata.parquet"

# Columns that benefit from dictionary encoding (low cardinality, many repeats)
CATEGORICAL_COLS = ("collection", "tax_group", "tf_family", "tf_class", "data_type")


def matrices_to_dataframe(records: list[dict]) -> pl.DataFrame:
    """
    Convert a list of JASPAR matrix records into a typed Polars DataFrame.

    The ``matrix_id`` field carries the version suffix (e.g. ``MA0106.3``).
    We split it into ``base_id`` (``MA0106``) and ``version`` (``3``) so that
    callers can filter to the latest version per TF.

    Parameters
    ----------
    records : list of dict
        Matrix metadata records as returned by :func:`fetch_all_matrices`.

    Returns
    -------
    pl.DataFrame
        One row per matrix with categorical low-cardinality columns,
        an ``Int32`` ``version`` column, and a ``Utf8`` ``base_id`` column.
    """
    # Build the DataFrame from the raw records — Polars infers dtypes row-wise.
    # ``strict=False`` lets us tolerate records that are missing optional keys
    # such as ``tf_family`` for older profiles.
    df = pl.DataFrame(records, strict=False)

    # Parse matrix_id -> (base_id, version). str.splitn returns a struct; we
    # extract the two fields into top-level columns.
    df = df.with_columns(
        pl.col("matrix_id")
          .str.splitn(".", 2)
          .struct.rename_fields(["base_id", "version_str"])
          .alias("_id_parts")
    ).unnest("_id_parts").with_columns(
        pl.col("version_str").cast(pl.Int32, strict=False).alias("version"),
    ).drop("version_str")

    # Cast categorical columns (skip silently if they are absent from this
    # particular query's response shape)
    cat_exprs = [
        pl.col(c).cast(pl.Categorical).alias(c)
        for c in CATEGORICAL_COLS
        if c in df.columns
    ]
    if cat_exprs:
        df = df.with_columns(cat_exprs)

    return df


matrix_df = matrices_to_dataframe(matrix_records)

# Persist to Parquet — columnar, compressed, preserves dtypes without re-parsing
matrix_df.write_parquet(METADATA_PARQUET)
print(f"Wrote {METADATA_PARQUET}  ({METADATA_PARQUET.stat().st_size / 1024:.1f} KB)")

print(f"\nShape: {matrix_df.shape[0]:,} rows × {matrix_df.shape[1]} columns")
print("\nSchema:")
for col, dtype in matrix_df.schema.items():
    print(f"  {col:<18} {dtype}")

matrix_df.head(5)